# Lecture 3 — Joining and Reshaping Data

This notebook demonstrates how to combine data from multiple sources and reshape
DataFrames between wide and long formats using pandas.

These operations are essential in business intelligence, where data typically comes from
several tables or systems that need to be combined before analysis.

**Credits:**

Created by Sippo Rossi for the course Python Programming for Business Intelligence at Hanken.

In [2]:
import pandas as pd

## 1. Sample data

We create two small DataFrames that represent tables from a retail company.
These mirror a typical scenario where customer data and order data are stored separately.

In [3]:
# Customers table
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Anna", "Erik", "Mia", "Olli"],
    "city": ["Helsinki", "Turku", "Tampere", "Oulu"]
})

customers

,customer_id,name,city
0,1,Anna,Helsinki
1,2,Erik,Turku
2,3,Mia,Tampere
3,4,Olli,Oulu


In [4]:
# Orders table
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 2, 3, 5],
    "product": ["Laptop", "Keyboard", "Monitor", "Desk Chair", "Mouse"],
    "amount": [899.99, 49.99, 349.99, 249.99, 19.99]
})

orders

,order_id,customer_id,product,amount
0,101,1,Laptop,899.99
1,102,2,Keyboard,49.99
2,103,2,Monitor,349.99
3,104,3,Desk Chair,249.99
4,105,5,Mouse,19.99


Note that customer 4 (Olli) has no orders, and order 105 refers to customer 5
who does not exist in the customers table. This lets us see the differences between join types.

## 2. Joining DataFrames with `merge()`

The `pd.merge()` function works like SQL joins. The `how` parameter controls the
join type.

### 2.1 Inner join

In [5]:
# Inner join: only rows where customer_id exists in both tables
inner = pd.merge(customers, orders, how="inner", on="customer_id")
inner

,customer_id,name,city,order_id,product,amount
0,1,Anna,Helsinki,101,Laptop,899.99
1,2,Erik,Turku,102,Keyboard,49.99
2,2,Erik,Turku,103,Monitor,349.99
3,3,Mia,Tampere,104,Desk Chair,249.99


### 2.2 Left join

In [6]:
# Left join: all customers, matched orders where available
left = pd.merge(customers, orders, how="left", on="customer_id")
left

,customer_id,name,city,order_id,product,amount
0,1,Anna,Helsinki,101.0,Laptop,899.99
1,2,Erik,Turku,102.0,Keyboard,49.99
2,2,Erik,Turku,103.0,Monitor,349.99
3,3,Mia,Tampere,104.0,Desk Chair,249.99
4,4,Olli,Oulu,NaN,NaN,NaN


### 2.3 Right join

In [7]:
# Right join: all orders, matched customers where available
right = pd.merge(customers, orders, how="right", on="customer_id")
right

,customer_id,name,city,order_id,product,amount
0,1,Anna,Helsinki,101,Laptop,899.99
1,2,Erik,Turku,102,Keyboard,49.99
2,2,Erik,Turku,103,Monitor,349.99
3,3,Mia,Tampere,104,Desk Chair,249.99
4,5,NaN,NaN,105,Mouse,19.99


### 2.4 Outer join

In [8]:
# Outer join: all rows from both tables
outer = pd.merge(customers, orders, how="outer", on="customer_id")
outer

,customer_id,name,city,order_id,product,amount
0,1,Anna,Helsinki,101.0,Laptop,899.99
1,2,Erik,Turku,102.0,Keyboard,49.99
2,2,Erik,Turku,103.0,Monitor,349.99
3,3,Mia,Tampere,104.0,Desk Chair,249.99
4,4,Olli,Oulu,NaN,NaN,NaN
5,5,NaN,NaN,105.0,Mouse,19.99


### 2.5 Merging on differently named columns

If the key columns have different names in the two DataFrames, use `left_on` and `right_on`.

In [9]:
# Example with different column names
orders_renamed = orders.rename(columns={"customer_id": "cust_id"})

merged = pd.merge(
    customers, orders_renamed,
    how="inner",
    left_on="customer_id",
    right_on="cust_id"
)
merged

,customer_id,name,city,order_id,cust_id,product,amount
0,1,Anna,Helsinki,101,1,Laptop,899.99
1,2,Erik,Turku,102,2,Keyboard,49.99
2,2,Erik,Turku,103,2,Monitor,349.99
3,3,Mia,Tampere,104,3,Desk Chair,249.99


## 3. Concatenating DataFrames with `concat()`

`pd.concat()` stacks DataFrames vertically (like SQL UNION) or horizontally.
This is useful when combining datasets that have the same columns.

In [12]:
# Orders from Q1 and Q2
q1_orders = pd.DataFrame({
    "order_id": [201, 202],
    "product": ["Laptop", "Mouse"],
    "amount": [899.99, 19.99]
})
print(q1_orders)
print()

q2_orders = pd.DataFrame({
    "order_id": [203, 204],
    "product": ["Keyboard", "Monitor"],
    "amount": [49.99, 349.99]
})
print(q2_orders)
print()

# Stack them vertically
all_orders = pd.concat([q1_orders, q2_orders], ignore_index=True)
all_orders

   order_id product  amount
0       201  Laptop  899.99
1       202   Mouse   19.99

   order_id   product  amount
0       203  Keyboard   49.99
1       204   Monitor  349.99



,order_id,product,amount
0,201,Laptop,899.99
1,202,Mouse,19.99
2,203,Keyboard,49.99
3,204,Monitor,349.99


## 4. Reshaping data: wide and long format

Data can be stored in two layouts:

- **Wide format**: each variable has its own column.
- **Long format**: one column contains variable names and another contains the values.

Different tools and analyses expect different formats, so converting between them is
a common task.

### 4.1 Sample data in wide format

In [13]:
# Quarterly revenue by region (wide format)
revenue_wide = pd.DataFrame({
    "region": ["North", "South", "East", "West"],
    "Q1": [12000, 15000, 9000, 11000],
    "Q2": [13500, 14000, 10500, 12500],
    "Q3": [14000, 16000, 11000, 13000],
    "Q4": [15500, 17500, 12000, 14500]
})

revenue_wide

,region,Q1,Q2,Q3,Q4
0,North,12000,13500,14000,15500
1,South,15000,14000,16000,17500
2,East,9000,10500,11000,12000
3,West,11000,12500,13000,14500


### 4.2 Wide to long with `melt()`

In [14]:
# Convert to long format
revenue_long = revenue_wide.melt(
    id_vars="region",
    var_name="quarter",
    value_name="revenue"
)

revenue_long

,region,quarter,revenue
0,North,Q1,12000
1,South,Q1,15000
2,East,Q1,9000
3,West,Q1,11000
4,North,Q2,13500
5,South,Q2,14000
6,East,Q2,10500
7,West,Q2,12500
8,North,Q3,14000
9,South,Q3,16000


### 4.3 Long to wide with `pivot()`

In [17]:
revenue_back = revenue_long.pivot(
    index="region",
    columns="quarter",
    values="revenue"
).rename_axis(columns=None).reset_index()

revenue_back

,region,Q1,Q2,Q3,Q4
0,East,9000,10500,11000,12000
1,North,12000,13500,14000,15500
2,South,15000,14000,16000,17500
3,West,11000,12500,13000,14500


### 4.4 When to use which format

| Format | Typical use case |
|---|---|
| Wide | Spreadsheet-style reporting, quick comparison across columns |
| Long | Plotting libraries (e.g. seaborn), groupby operations, statistical modelling |

## 5. Putting it together: a practical example

Suppose you receive monthly sales data from three stores as separate DataFrames and
need to combine them, enrich with store metadata, and reshape for reporting.

In [18]:
# Sales data from three stores
store_a = pd.DataFrame({
    "store_id": [1, 1, 1],
    "month": ["Jan", "Feb", "Mar"],
    "sales": [45000, 48000, 52000]
})

store_b = pd.DataFrame({
    "store_id": [2, 2, 2],
    "month": ["Jan", "Feb", "Mar"],
    "sales": [32000, 34000, 31000]
})

store_c = pd.DataFrame({
    "store_id": [3, 3, 3],
    "month": ["Jan", "Feb", "Mar"],
    "sales": [28000, 30000, 33000]
})

# Store metadata
stores = pd.DataFrame({
    "store_id": [1, 2, 3],
    "city": ["Helsinki", "Espoo", "Vantaa"]
})

In [19]:
# Step 1: Combine the sales data
all_sales = pd.concat([store_a, store_b, store_c], ignore_index=True)
all_sales

,store_id,month,sales
0,1,Jan,45000
1,1,Feb,48000
2,1,Mar,52000
3,2,Jan,32000
4,2,Feb,34000
5,2,Mar,31000
6,3,Jan,28000
7,3,Feb,30000
8,3,Mar,33000


In [20]:
# Step 2: Join with store metadata
sales_with_city = pd.merge(all_sales, stores, on="store_id")
sales_with_city

,store_id,month,sales,city
0,1,Jan,45000,Helsinki
1,1,Feb,48000,Helsinki
2,1,Mar,52000,Helsinki
3,2,Jan,32000,Espoo
4,2,Feb,34000,Espoo
5,2,Mar,31000,Espoo
6,3,Jan,28000,Vantaa
7,3,Feb,30000,Vantaa
8,3,Mar,33000,Vantaa


In [21]:
# Step 3: Pivot to a wide report (cities as rows, months as columns)
report = sales_with_city.pivot(
    index="city",
    columns="month",
    values="sales"
)[["Jan", "Feb", "Mar"]]  # Reorder columns

report

month,Jan,Feb,Mar
city,,,
Espoo,32000,34000,31000
Helsinki,45000,48000,52000
Vantaa,28000,30000,33000
